# Практика 4. Чищення набору даних

Мета заняття: побудувати «брудний» набір даних із типовими дефектами та послідовно усунути пропуски, дублікати, неправильні типи, неузгоджені категорії й викид.

**Інструкція:** у першій Python-клітинці змініть `city`, якщо ваш номер варіанта відповідає іншому місту. За замовчуванням обрано Київ, варіант 1.

In [2]:
import numpy as np
import pandas as pd

variants = {
    'Київ': {'variant': 1, 'base_price': 850, 'base_quantity': 3},
    'Львів': {'variant': 2, 'base_price': 720, 'base_quantity': 4},
    'Одеса': {'variant': 3, 'base_price': 640, 'base_quantity': 5},
    'Харків': {'variant': 4, 'base_price': 780, 'base_quantity': 2},
    'Дніпро': {'variant': 5, 'base_price': 810, 'base_quantity': 3},
    'Чернігів': {'variant': 6, 'base_price': 690, 'base_quantity': 4},
    'Івано-Франківськ': {'variant': 7, 'base_price': 760, 'base_quantity': 3},
    'Полтава': {'variant': 8, 'base_price': 700, 'base_quantity': 4},
    'Суми': {'variant': 9, 'base_price': 730, 'base_quantity': 2},
    'Ужгород': {'variant': 10, 'base_price': 670, 'base_quantity': 5}
}

city = 'Київ'
if city not in variants:
    raise ValueError(f'Невідоме місто: {city}. Оберіть одне з: {list(variants)}')
base = variants[city]
base_price = base['base_price']
base_quantity = base['base_quantity']
variant_number = base['variant']
product = f'Товар варіанта {variant_number}'


orders_dirty = pd.DataFrame([
    {'дата': '2026-09-01', 'товар': product, 'місто': city,       'ціна': base_price,           'кількість': base_quantity},
    {'дата': '2026-09-02', 'товар': product, 'місто': f' {city} ', 'ціна': f'{base_price} грн',    'кількість': 2},
    {'дата': '2026-09-03', 'товар': product, 'місто': city.upper(), 'ціна': f'{base_price} грн',    'кількість': np.nan},
    {'дата': '2026-09-04', 'товар': product, 'місто': city,       'ціна': base_price * 10,       'кількість': 4},
    {'дата': '2026-09-05', 'товар': product, 'місто': city,       'ціна': base_price,           'кількість': np.nan},
    {'дата': '2026-09-06', 'товар': product, 'місто': f' {city} ', 'ціна': base_price,           'кількість': 1},
    {'дата': '2026-09-07', 'товар': product, 'місто': city.upper(), 'ціна': base_price,           'кількість': 5},
    {'дата': '2026-09-07', 'товар': product, 'місто': city.upper(), 'ціна': base_price,           'кількість': 5}
])

print(f'Варіант {variant_number}: {city}; базова ціна: {base_price} грн; базова кількість: {base_quantity}')
display(orders_dirty)
print('Типи даних до очищення:')
display(orders_dirty.dtypes.to_frame('Тип'))

Варіант 1: Київ; базова ціна: 850 грн; базова кількість: 3


,дата,товар,місто,ціна,кількість
0,2026-09-01,Товар варіанта 1,Київ,850,3.0
1,2026-09-02,Товар варіанта 1,Київ,850 грн,2.0
2,2026-09-03,Товар варіанта 1,КИЇВ,850 грн,NaN
3,2026-09-04,Товар варіанта 1,Київ,8500,4.0
4,2026-09-05,Товар варіанта 1,Київ,850,NaN
5,2026-09-06,Товар варіанта 1,Київ,850,1.0
6,2026-09-07,Товар варіанта 1,КИЇВ,850,5.0
7,2026-09-07,Товар варіанта 1,КИЇВ,850,5.0


Типи даних до очищення:


,Тип
дата,str
товар,str
місто,str
ціна,object
кількість,float64


## Завдання 1. Побудова брудного набору

У `orders_dirty` навмисно присутні всі потрібні дефекти: два пропуски в `кількість`, один повний дублікат рядка, щонайменше два значення `ціна` як рядки з `грн`, три варіанти назви міста та одне значення ціни, що у 10 разів перевищує базову. Наступна клітинка перевіряє це явно.

In [3]:
assert orders_dirty['кількість'].isna().sum() == 2
assert orders_dirty.duplicated().sum() == 1
assert orders_dirty['ціна'].map(lambda value: isinstance(value, str)).sum() >= 2
assert orders_dirty['місто'].nunique() == 3
assert orders_dirty['ціна'].map(lambda value: value == base_price * 10).sum() == 1
print('Перевірка дефектів пройдена: усі п’ять типів присутні.')

Перевірка дефектів пройдена: усі п’ять типів присутні.


## Завдання 2. Пропуски

In [4]:
missing_before = orders_dirty.isna().sum()
display(missing_before.to_frame('Кількість пропусків'))

quantity_median = orders_dirty['кількість'].median()
orders_clean = orders_dirty.copy()
orders_clean['кількість'] = orders_clean['кількість'].fillna(quantity_median)
print(f'Медіана кількості: {quantity_median}')
print('Пропуски в кількості заповнено медіаною.')
display(orders_clean)

,Кількість пропусків
дата,0
товар,0
місто,0
ціна,0
кількість,2


Медіана кількості: 3.5
Пропуски в кількості заповнено медіаною.


,дата,товар,місто,ціна,кількість
0,2026-09-01,Товар варіанта 1,Київ,850,3.0
1,2026-09-02,Товар варіанта 1,Київ,850 грн,2.0
2,2026-09-03,Товар варіанта 1,КИЇВ,850 грн,3.5
3,2026-09-04,Товар варіанта 1,Київ,8500,4.0
4,2026-09-05,Товар варіанта 1,Київ,850,3.5
5,2026-09-06,Товар варіанта 1,Київ,850,1.0
6,2026-09-07,Товар варіанта 1,КИЇВ,850,5.0
7,2026-09-07,Товар варіанта 1,КИЇВ,850,5.0


Медіана доречніша за середнє, тому що в наборі лише 8 спостережень, а одне замовлення має аномально велику ціну і загалом дані ще потребують перевірки. Медіана стійкіша до крайніх значень і не зміщує типовий розмір замовлення так сильно, як середнє.

## Завдання 3. Дублікати

In [5]:
full_duplicates = orders_clean[orders_clean.duplicated(keep=False)]
print(f'Повні дублікати без subset: {orders_clean.duplicated().sum()}')
display(full_duplicates)

# До перевірки за змістовними полями нормалізуємо місто.
orders_clean['місто'] = orders_clean['місто'].str.strip().str.lower()
duplicate_key = ['дата', 'товар', 'місто']
print(f'Дублікати за полями {duplicate_key}: {orders_clean.duplicated(subset=duplicate_key).sum()}')

orders_clean = orders_clean.drop_duplicates(subset=duplicate_key, keep='first').reset_index(drop=True)
print(f'Після видалення дубліката залишилося рядків: {len(orders_clean)}')
display(orders_clean)

Повні дублікати без subset: 1


,дата,товар,місто,ціна,кількість
6,2026-09-07,Товар варіанта 1,КИЇВ,850,5.0
7,2026-09-07,Товар варіанта 1,КИЇВ,850,5.0


Дублікати за полями ['дата', 'товар', 'місто']: 1
Після видалення дубліката залишилося рядків: 7


,дата,товар,місто,ціна,кількість
0,2026-09-01,Товар варіанта 1,київ,850,3.0
1,2026-09-02,Товар варіанта 1,київ,850 грн,2.0
2,2026-09-03,Товар варіанта 1,київ,850 грн,3.5
3,2026-09-04,Товар варіанта 1,київ,8500,4.0
4,2026-09-05,Товар варіанта 1,київ,850,3.5
5,2026-09-06,Товар варіанта 1,київ,850,1.0
6,2026-09-07,Товар варіанта 1,київ,850,5.0


`duplicated()` без `subset` шукає повний збіг усіх значень у рядку. `duplicated(subset=...)` перевіряє лише вказані поля, тому може знайти записи, які відрізняються другорядним полем, але описують те саме замовлення. Тут замовлення вважаємо унікальним за датою, товаром і містом.

## Завдання 4. Типи даних і узгодження категорій

In [6]:
orders_clean['ціна'] = (
    orders_clean['ціна'].astype(str).str.replace(' грн', '', regex=False).astype(float)
)
city_mapping = {
    city.strip().lower(): city,
    city.upper().strip().lower(): city
}
orders_clean['місто'] = orders_clean['місто'].replace(city_mapping)

print('Унікальні назви міста після узгодження:', orders_clean['місто'].unique())
print('Типи після очищення:')
display(orders_clean.dtypes.to_frame('Тип'))
display(orders_clean)

Унікальні назви міста після узгодження: <StringArray>
['Київ']
Length: 1, dtype: str
Типи після очищення:


,Тип
дата,str
товар,str
місто,str
ціна,float64
кількість,float64


,дата,товар,місто,ціна,кількість
0,2026-09-01,Товар варіанта 1,Київ,850.0,3.0
1,2026-09-02,Товар варіанта 1,Київ,850.0,2.0
2,2026-09-03,Товар варіанта 1,Київ,850.0,3.5
3,2026-09-04,Товар варіанта 1,Київ,8500.0,4.0
4,2026-09-05,Товар варіанта 1,Київ,850.0,3.5
5,2026-09-06,Товар варіанта 1,Київ,850.0,1.0
6,2026-09-07,Товар варіанта 1,Київ,850.0,5.0


## Завдання 5. Викиди

In [7]:
q1, q3 = orders_clean['ціна'].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_mask = (orders_clean['ціна'] < lower_bound) | (orders_clean['ціна'] > upper_bound)

print(f'Q1 = {q1:.2f} грн; Q3 = {q3:.2f} грн; IQR = {iqr:.2f} грн')
print(f'Межі IQR: [{lower_bound:.2f}; {upper_bound:.2f}] грн')
print('Підозрілі значення:')
display(orders_clean.loc[outlier_mask, ['дата', 'ціна', 'кількість']])

orders_clean['ціна'].describe().to_frame('Описова статистика')

Q1 = 850.00 грн; Q3 = 850.00 грн; IQR = 0.00 грн
Межі IQR: [850.00; 850.00] грн
Підозрілі значення:


,дата,ціна,кількість
3,2026-09-04,8500.0,4.0


,Описова статистика
count,7.000000
mean,1942.857143
std,2891.428219
min,850.000000
25%,850.000000
50%,850.000000
75%,850.000000
max,8500.000000


У цьому наборі велике значення ціни я вважаю помилкою введення: воно рівно у 10 разів більше за базову ціну, а всі інші замовлення мають ціну на рівні базової. Найімовірніше, це зайвий нуль. Для реального великого замовлення потрібні додаткові підтвердження, наприклад більша кількість товару або окремий тип замовлення; тому зараз значення слід виправити після перевірки джерела, а не безумовно залишати.

## Короткі відповіді на питання для здачі

1. **Чому середнє може спотворити стандартне відхилення?** Заповнення пропуску середнім додає штучне значення в центрі розподілу, зменшує розкид і може занизити стандартне відхилення. Медіана менше залежить від крайніх значень, тому для малого й потенційно асиметричного набору часто надійніша.
2. **Чим відрізняється `duplicated()` без `subset` від варіанта з `subset`?** Без `subset` потрібен повний збіг рядка. З `subset` збіг перевіряється лише за полями, які визначають унікальність об'єкта, тому можна знайти логічні дублікати, навіть якщо другорядні значення різняться.
3. **Чому не можна автоматично видаляти все поза межами IQR?** IQR лише позначає статистично нетипове значення, але не доводить, що воно є помилкою. Викид може бути реальною рідкісною подією, тому його треба перевірити за предметним контекстом і джерелом даних.

## Підсумок

У роботі побудовано набір із пропусками, дублікатом, текстовими цінами, різними написаннями міста та великим підозрілим значенням. Дефекти усунуто послідовно: пропуски заповнено медіаною, дублікат видалено за ключем замовлення, ціну перетворено на `float`, назву міста узгоджено, а викид проаналізовано за правилом IQR і предметним контекстом.